# GN_DW_POC 데이터 적재 노트북
Excel 파일(스테이지) → RAW 테이블 18개 → ANALYTICS 뷰 13개

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS GN_DW_POC;
CREATE SCHEMA IF NOT EXISTS GN_DW_POC.RAW;
CREATE SCHEMA IF NOT EXISTS GN_DW_POC.ANALYTICS;
CREATE STAGE IF NOT EXISTS GN_DW_POC.RAW.STG_POC_DATA;

In [ ]:
!pip install --no-index et_xmlfile-2.0.0-py3-none-any.whl
!pip install --no-index openpyxl-3.1.5-py2.py3-none-any.whl

In [ ]:
import subprocess, sys, glob, os

import pandas as pd
import openpyxl
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import *
import io, tempfile

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE GN_DW_POC").collect()
session.sql("USE SCHEMA RAW").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

STAGE = '@GN_DW_POC.RAW.STG_POC_DATA'

def download_from_stage(filename):
    tmp_dir = tempfile.mkdtemp()
    session.file.get(f"{STAGE}/{filename}", tmp_dir)
    local_path = os.path.join(tmp_dir, filename)
    if not os.path.exists(local_path):
        files = glob.glob(os.path.join(tmp_dir, '*'))
        if files:
            local_path = files[0]
    return local_path

def cs(df, cols):
    for col in cols:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace({'None': None, 'nan': None, 'NaT': None, 'nat': None})
    return df

def clean_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        mask = df[col].isna()
        df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S')
        df.loc[mask, col] = None
    return df

def load_to_table(df, table_name, col_defs):
    session.sql(f'DROP TABLE IF EXISTS GN_DW_POC.RAW.{table_name}').collect()
    session.sql(f'CREATE TABLE GN_DW_POC.RAW.{table_name} ({col_defs})').collect()
    sp_df = session.create_dataframe(df)
    sp_df.write.mode('append').save_as_table(f'GN_DW_POC.RAW.{table_name}')
    cnt = session.sql(f'SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.{table_name}').collect()[0]['CNT']
    print(f"  {table_name}: {cnt:,} rows")
    return cnt

print("Setup complete")

In [ ]:
print("=== Phase 1: 공통/마스터 테이블 4개 ===")

print("\n[T01] DIM_CAMPAIGN_CODE")
fp = download_from_stage('공통_캠페인코드.xlsx')
df = pd.read_excel(fp, sheet_name='공통캠페인코드')
kcols = ['브랜드코드', '브랜드명', '브랜드사용여부', '브랜드사용부서', '상위캠페인코드', '상위캠페인명',
         '상위캠페인사용여부', '세부캠페인코드', '세부캠페인명', '세부캠페인사용부서(실적부서)',
         '세부캠페인사용여부', '세부캠페인시작일']
df.columns = kcols
df = cs(df, kcols)
col_defs = ', '.join([f'"{ c }" VARCHAR' for c in kcols])
load_to_table(df, 'DIM_CAMPAIGN_CODE', col_defs)

print("\n[T02] DIM_MEMBER_ATTRIBUTE")
fp = download_from_stage('공통_회원특성.xlsx')
df = pd.read_excel(fp, sheet_name='회원특성')
kcols = ['회원번호', '성별', '연령대', '지역']
df.columns = kcols
df = cs(df, kcols)
col_defs = ', '.join([f'"{ c }" VARCHAR' for c in kcols])
load_to_table(df, 'DIM_MEMBER_ATTRIBUTE', col_defs)

print("\nPhase 1-A (DIM tables) complete")

In [ ]:
print("[T03] FACT_MEMBER_DEV_ALL")
fp = download_from_stage('공통_회원개발이력_2025.xlsx')
df = pd.read_excel(fp, sheet_name='25년 전체개발이력(신규, 증액, 감액, 재후원, 중단)')
kcols = ['순번', '법인구분', '신청일', '실적부서', '브랜드', '홍보방법', '가입경로', '상위캠페인',
         '(가입)캠페인', '후원사업', '개발구분(후원중단/재후원/신규/증액/감액)', '회원번호', '금액',
         '중단일', '중단사유', '후원번호', '후원사업번호']
df.columns = kcols
str_cols = ['법인구분', '실적부서', '브랜드', '홍보방법', '가입경로', '상위캠페인', '(가입)캠페인',
            '후원사업', '개발구분(후원중단/재후원/신규/증액/감액)', '회원번호', '중단사유']
df = cs(df, str_cols)
df = clean_dates(df, ['중단일'])
for nc in ['순번', '금액', '후원번호', '후원사업번호']:
    df[nc] = pd.to_numeric(df[nc], errors='coerce')
col_defs = ('"순번" NUMBER, "법인구분" VARCHAR, "신청일" NUMBER, "실적부서" VARCHAR, '
            '"브랜드" VARCHAR, "홍보방법" VARCHAR, "가입경로" VARCHAR, "상위캠페인" VARCHAR, '
            '"(가입)캠페인" VARCHAR, "후원사업" VARCHAR, "개발구분(후원중단/재후원/신규/증액/감액)" VARCHAR, '
            '"회원번호" VARCHAR, "금액" NUMBER, "중단일" VARCHAR, "중단사유" VARCHAR, '
            '"후원번호" NUMBER, "후원사업번호" NUMBER')
load_to_table(df, 'FACT_MEMBER_DEV_ALL', col_defs)

print("\n[T04] FACT_MEMBER_DEV_NEW")
df = pd.read_excel(fp, sheet_name='25년 신규, 증액, 재후원 개발건에 대한 이력')
kcols = ['순번', '법인구분', '신청일', '실적부서', '브랜드', '홍보방법', '가입경로', '상위캠페인',
         '(가입)캠페인', '후원사업', '회원번호', '개발금액', '25년 총 납입금액',
         '중단일', '중단사유', '후원번호', '후원사업번호']
df.columns = kcols
str_cols = ['법인구분', '실적부서', '브랜드', '홍보방법', '가입경로', '상위캠페인', '(가입)캠페인',
            '후원사업', '회원번호', '중단사유']
df = cs(df, str_cols)
df = clean_dates(df, ['중단일'])
for nc in ['순번', '개발금액', '25년 총 납입금액', '후원번호', '후원사업번호']:
    df[nc] = pd.to_numeric(df[nc], errors='coerce')
col_defs = ('"순번" NUMBER, "법인구분" VARCHAR, "신청일" NUMBER, "실적부서" VARCHAR, '
            '"브랜드" VARCHAR, "홍보방법" VARCHAR, "가입경로" VARCHAR, "상위캠페인" VARCHAR, '
            '"(가입)캠페인" VARCHAR, "후원사업" VARCHAR, "회원번호" VARCHAR, '
            '"개발금액" NUMBER, "25년 총 납입금액" NUMBER, "중단일" VARCHAR, "중단사유" VARCHAR, '
            '"후원번호" NUMBER, "후원사업번호" NUMBER')
load_to_table(df, 'FACT_MEMBER_DEV_NEW', col_defs)

print("\nPhase 1 complete")

In [ ]:
print("=== Phase 2: 마케팅 대행사 테이블 6개 ===")

print("\n[T05] FACT_DRTV_MONTHLY_DEV")
fp = download_from_stage('마케팅_25년_DRTV대행사.xlsx')
df = pd.read_excel(fp, sheet_name='영상광고 월별 개발 현황')
df = df.iloc[:, :16].copy()
kcols = ['구분', '월별 목표', '월별 실적', '달성율', '누계목표', '누계실적', '누계달성율',
         '월별 예산', '집행 예산', '누계집행율', '누계 집행 예산', '연목표', '연 광고예산',
         '해당연도', '예산절차', '월 구분']
df.columns = kcols
df = clean_dates(df, ['구분'])
for c in ['달성율', '누계달성율', '누계집행율']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = cs(df, ['해당연도', '예산절차', '월 구분'])
col_defs = ('"구분" VARCHAR, "월별 목표" FLOAT, "월별 실적" FLOAT, "달성율" FLOAT, '
            '"누계목표" FLOAT, "누계실적" FLOAT, "누계달성율" FLOAT, "월별 예산" FLOAT, '
            '"집행 예산" FLOAT, "누계집행율" FLOAT, "누계 집행 예산" FLOAT, "연목표" FLOAT, '
            '"연 광고예산" FLOAT, "해당연도" VARCHAR, "예산절차" VARCHAR, "월 구분" VARCHAR')
load_to_table(df, 'FACT_DRTV_MONTHLY_DEV', col_defs)

print("\n[T06] FACT_DRTV_BROADCAST_EFF")
df = pd.read_excel(fp, sheet_name='월별 편성 및 효율')
kcols = ['채널', '요일', '방송일자', '시간대', '주중/토/일', '프로그램 시작시간', '편성명',
         'CM', 'CM위치', '광고 시작시간', '광고시청률', '광고종료시간', 'Spot Type',
         '횟수', '초수', '실구매광고비(원)', '인입콜', 'CPC', '소재',
         'CRM(세부캠페인)명칭', '소재 일관화', '주차', '요일2', '방송월', '채널사 유형', '해당연도']
df.columns = kcols
df = cs(df, kcols)
col_defs = ', '.join([f'"{ c }" VARCHAR' for c in kcols])
load_to_table(df, 'FACT_DRTV_BROADCAST_EFF', col_defs)

print("\nPhase 2-A (DRTV) complete")

In [ ]:
print("[T07] FACT_DIGITAL_MONTHLY_DEV")
fp = download_from_stage('마케팅_25년_디지털대행사.xlsx')
df = pd.read_excel(fp, sheet_name='디지털 월별 개발 현황')
kcols = ['연도', '예산절차', '월', '날짜', '월별 개발목표(건)', '월별 개발실적(건)',
         '연목표(건)', '월별 편성예산(원)', '월별 집행예산(원)', '연 광고 예산(원)']
df.columns = kcols
df = cs(df, ['연도', '예산절차', '월', '날짜'])
col_defs = ('"연도" VARCHAR, "예산절차" VARCHAR, "월" VARCHAR, "날짜" VARCHAR, '
            '"월별 개발목표(건)" FLOAT, "월별 개발실적(건)" FLOAT, "연목표(건)" FLOAT, '
            '"월별 편성예산(원)" FLOAT, "월별 집행예산(원)" FLOAT, "연 광고 예산(원)" FLOAT')
load_to_table(df, 'FACT_DIGITAL_MONTHLY_DEV', col_defs)

print("\n[T08] FACT_DIGITAL_AD_DETAIL")
df = pd.read_excel(fp, sheet_name='2025년 사단법인 광고운영상세')
kcols = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
         '날짜', '주차', '일자', '요일', '캠페인명', '소재', '노출수', '클릭수',
         'GA 광고비', 'GA 전환명수', 'GA 개발건수', '상위캠페인명']
df.columns = kcols
df = clean_dates(df, ['날짜'])
for nc in ['노출수', '클릭수', 'GA 광고비', 'GA 전환명수', 'GA 개발건수']:
    df[nc] = pd.to_numeric(df[nc], errors='coerce')
str_cols = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
            '주차', '일자', '요일', '캠페인명', '소재', '상위캠페인명']
df = cs(df, str_cols)
col_defs = ('"연도" VARCHAR, "국내/해외" VARCHAR, "사업/사례" VARCHAR, "캠페인 유형" VARCHAR, '
            '"광고유형" VARCHAR, "월" VARCHAR, "기기" VARCHAR, "매체" VARCHAR, '
            '"날짜" VARCHAR, "주차" VARCHAR, "일자" VARCHAR, "요일" VARCHAR, '
            '"캠페인명" VARCHAR, "소재" VARCHAR, "노출수" FLOAT, "클릭수" FLOAT, '
            '"GA 광고비" FLOAT, "GA 전환명수" FLOAT, "GA 개발건수" FLOAT, "상위캠페인명" VARCHAR')
load_to_table(df, 'FACT_DIGITAL_AD_DETAIL', col_defs)

print("\nPhase 2-B (Digital) complete")

In [ ]:
print("[T09] FACT_RETRANSMIT_MONTHLY_DEV")
fp = download_from_stage('마케팅_25년_재송출대행사.xlsx')
df = pd.read_excel(fp, sheet_name='재송출 월별 개발 현황')
df = df.iloc[:, :18].copy()
kcols = ['구분', '월별 목표', '월별 실적', '달성율', '누계목표', '누계실적', '누계달성율',
         '월별 예산', '집행 예산', '누계집행율', '누계 집행 예산', '연목표', '연 광고예산',
         '연도', '예산절차', '월 구분', '방송구분', '실적부서']
df.columns = kcols
df = clean_dates(df, ['구분'])
for c in ['달성율', '누계달성율', '누계집행율']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = cs(df, ['연도', '예산절차', '월 구분', '방송구분', '실적부서'])
col_defs = ('"구분" VARCHAR, "월별 목표" FLOAT, "월별 실적" FLOAT, "달성율" FLOAT, '
            '"누계목표" FLOAT, "누계실적" FLOAT, "누계달성율" FLOAT, "월별 예산" FLOAT, '
            '"집행 예산" FLOAT, "누계집행율" FLOAT, "누계 집행 예산" FLOAT, "연목표" FLOAT, '
            '"연 광고예산" FLOAT, "연도" VARCHAR, "예산절차" VARCHAR, "월 구분" VARCHAR, '
            '"방송구분" VARCHAR, "실적부서" VARCHAR')
load_to_table(df, 'FACT_RETRANSMIT_MONTHLY_DEV', col_defs)

print("\n[T10] FACT_RETRANSMIT_BROADCAST_CONV")
df = pd.read_excel(fp, sheet_name='방송 송출 및 전환현황')
kcols = ['구분', '년도', '방송월', '방송사', '방송명', '상위캠페인명', '본방송 구분',
         '날짜', '요일', '방송시간', '인입콜', '회원개발(건)', '방송편성비',
         '주차', '횟수', '시간대 구분', '국내/해외 구분', '실적부서', '방송사 유형', '방송 대분류']
df.columns = kcols
df = clean_dates(df, ['날짜'])
str_cols = ['구분', '년도', '방송월', '방송사', '방송명', '상위캠페인명', '본방송 구분',
            '요일', '방송시간', '주차', '시간대 구분', '국내/해외 구분', '실적부서',
            '방송사 유형', '방송 대분류']
df = cs(df, str_cols)
col_defs = ('"구분" VARCHAR, "년도" VARCHAR, "방송월" VARCHAR, "방송사" VARCHAR, '
            '"방송명" VARCHAR, "상위캠페인명" VARCHAR, "본방송 구분" VARCHAR, '
            '"날짜" VARCHAR, "요일" VARCHAR, "방송시간" VARCHAR, "인입콜" FLOAT, '
            '"회원개발(건)" FLOAT, "방송편성비" FLOAT, "주차" VARCHAR, "횟수" FLOAT, '
            '"시간대 구분" VARCHAR, "국내/해외 구분" VARCHAR, "실적부서" VARCHAR, '
            '"방송사 유형" VARCHAR, "방송 대분류" VARCHAR')
load_to_table(df, 'FACT_RETRANSMIT_BROADCAST_CONV', col_defs)

print("\nPhase 2 complete")

In [ ]:
print("=== Phase 3: 회원/SMS 테이블 3개 ===")

print("\n[T11] FACT_DISCONTINUED_MEMBER")
fp = download_from_stage('회원_중단회원목록_2025.xlsx')
df = pd.read_excel(fp, sheet_name='중단회원목록_20250101_20251231')
kcols = ['순번', '법인구분', '회원번호', '회원구분', '납입방식', '후원금액', '가입일', '중단일',
         '중단사유', '가입캠페인(세부캠페인)', '브랜드', '상위캠페인', '가입부서(실적부서)']
df.columns = kcols
str_cols = ['법인구분', '회원번호', '회원구분', '납입방식', '중단사유',
            '가입캠페인(세부캠페인)', '브랜드', '상위캠페인', '가입부서(실적부서)']
df = cs(df, str_cols)
col_defs = ('"순번" NUMBER, "법인구분" VARCHAR, "회원번호" VARCHAR, "회원구분" VARCHAR, '
            '"납입방식" VARCHAR, "후원금액" NUMBER, "가입일" NUMBER, "중단일" NUMBER, '
            '"중단사유" VARCHAR, "가입캠페인(세부캠페인)" VARCHAR, "브랜드" VARCHAR, '
            '"상위캠페인" VARCHAR, "가입부서(실적부서)" VARCHAR')
load_to_table(df, 'FACT_DISCONTINUED_MEMBER', col_defs)

print("\n[T12] FACT_MARKETING_SEND_NEW")
fp = download_from_stage('회원_신규증액재후원 마케팅_발송.xlsx')
df = pd.read_excel(fp, sheet_name='문자알림톡발송내역일괄')
kcols = ['순번', '발송구분(대)', '발송구분(중)', '발송구분(소)', '제목', '총건수', '성공건수',
         '회원번호', '발송일시', '발송상태', '대체발송', '등록일', '성공률(%)', '브랜드', '상위캠페인']
df.columns = kcols
df = cs(df, [c for c in kcols if c != '순번'])
col_defs = '"순번" NUMBER, ' + ', '.join([f'"{ c }" VARCHAR' for c in kcols if c != '순번'])
load_to_table(df, 'FACT_MARKETING_SEND_NEW', col_defs)

print("\nPhase 3-A (Member tables) complete")

In [ ]:
print("[T13] FACT_SMS_ALIMTALK_SEND (13 files)")
sms_kcols = ['순번', '발송구분(대)', '발송구분(중)', '발송구분(소)', '제목', '총건수', '성공건수',
             '회원번호', '발송일시', '발송상태', '대체발송', '등록일', '성공률(%)']
col_defs = '"순번" NUMBER, ' + ', '.join([f'"{ c }" VARCHAR' for c in sms_kcols if c != '순번'])
session.sql(f'DROP TABLE IF EXISTS GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND').collect()
session.sql(f'CREATE TABLE GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND ({col_defs})').collect()

total = 0
for i in range(1, 14):
    fname = f'문자_알림톡발송내역_{i}.xlsx'
    sname = f'문자_알림톡발송내역_{i}'
    print(f"  File {i}/13: {fname}...", end='', flush=True)
    fp = download_from_stage(fname)
    df = pd.read_excel(fp, sheet_name=sname)
    df.columns = sms_kcols
    df = cs(df, [c for c in sms_kcols if c != '순번'])
    sp_df = session.create_dataframe(df)
    sp_df.write.mode('append').save_as_table('GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND')
    total += len(df)
    print(f" {len(df):,} rows (누적: {total:,})")

print(f"\nFACT_SMS_ALIMTALK_SEND total: {total:,} rows")
print("Phase 3 complete")

In [ ]:
print("=== Phase 4: GA 데이터 테이블 5개 ===")

fp = download_from_stage('회원_GA데이터.xlsx')

sheets_config = {
    '홈페이지_나의후원 전체 방문수': {
        'table': 'FACT_GA_VISITS_TOTAL',
        'skip_rows': 6,
        'dim_cols': ['페이지경로', '이벤트이름', '세션캠페인', '회원ID'],
        'metric_cols': ['세션수', '페이지뷰', '활성사용자수', '방문수', '이벤트수'],
    },
    '홈페이지_나의후원 PC 방문수': {
        'table': 'FACT_GA_VISITS_PC',
        'skip_rows': 6,
        'dim_cols': ['페이지경로', '회원ID'],
        'metric_cols': ['세션수', '페이지뷰', '활성사용자수', '방문수', '이벤트수'],
    },
    '홈페이지_나의후원 M 방문수': {
        'table': 'FACT_GA_VISITS_MOBILE',
        'skip_rows': 6,
        'dim_cols': ['페이지경로', '이벤트이름', '회원ID'],
        'metric_cols': ['세션수', '페이지뷰', '활성사용자수', '방문수', '이벤트수'],
    },
    '홈페이지_나의후원 APP 방문수': {
        'table': 'FACT_GA_VISITS_APP',
        'skip_rows': 6,
        'dim_cols': ['페이지경로', '이벤트이름', '회원ID'],
        'metric_cols': ['세션수', '페이지뷰', '활성사용자수', '방문수', '이벤트수'],
    },
    '26 신규증액 피드백 서비스페이지 회원번호URL별 유입': {
        'table': 'FACT_GA_FEEDBACK_PAGE',
        'skip_rows': 6,
        'dim_cols': ['페이지경로쿼리'],
        'metric_cols': ['세션수', '페이지뷰', '이벤트수', '이탈률', '참여율', '평균세션시간'],
    },
}

for sname, cfg in sheets_config.items():
    tname = cfg['table']
    print(f"\n=== {tname} ({sname}) ===")
    
    wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
    ws = wb[sname]
    all_rows = [list(r) for r in ws.iter_rows(values_only=True)]
    wb.close()
    
    skip = cfg['skip_rows']
    dim_cols = cfg['dim_cols']
    metric_cols = cfg['metric_cols']
    n_dim = len(dim_cols)
    n_metric = len(metric_cols)
    total_cols = dim_cols + metric_cols
    
    data_rows = all_rows[skip + 1:]
    parsed = []
    for r in data_rows:
        if not r or all(v is None for v in r):
            continue
        row_vals = list(r)
        n_with_total = n_dim + n_metric + 1
        while len(row_vals) < n_with_total:
            row_vals.append(None)
        dim_data = row_vals[:n_dim]
        metric_data = row_vals[n_dim:n_dim + n_metric]
        parsed.append(dim_data + metric_data)
    
    if not parsed:
        print(f"  -> 데이터 없음, 스킵")
        continue
    
    df = pd.DataFrame(parsed, columns=total_cols)
    df = cs(df, df.columns.tolist())
    col_defs = ', '.join([f'"{ c }" VARCHAR' for c in df.columns])
    load_to_table(df, tname, col_defs)

print("\nPhase 4 complete")

In [ ]:
%%sql -r raw_tables_check
SELECT TABLE_NAME, ROW_COUNT
FROM GN_DW_POC.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'RAW' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME

In [ ]:
%%sql -r v1_result
-- ① V_MEMBER_DEV_STATUS: 회원개발현황 (목표 vs 실적, 매체 통합)
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_MEMBER_DEV_STATUS AS
WITH drtv AS (
    SELECT
        'DRTV' AS "매체구분",
        "구분" AS "기준일",
        "월 구분" AS "월",
        "월별 목표",
        "월별 실적",
        "달성율",
        "누계목표",
        "누계실적",
        "누계달성율",
        "해당연도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_DRTV_MONTHLY_DEV
),
digital AS (
    SELECT
        '디지털' AS "매체구분",
        "날짜" AS "기준일",
        "월",
        "월별 개발목표(건)" AS "월별 목표",
        "월별 개발실적(건)" AS "월별 실적",
        CASE WHEN "월별 개발목표(건)" > 0 THEN "월별 개발실적(건)" / "월별 개발목표(건)" ELSE NULL END AS "달성율",
        NULL AS "누계목표",
        NULL AS "누계실적",
        NULL AS "누계달성율",
        "연도"
    FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV
),
retransmit AS (
    SELECT
        '재송출' AS "매체구분",
        "구분" AS "기준일",
        "월 구분" AS "월",
        "월별 목표",
        "월별 실적",
        "달성율",
        "누계목표",
        "누계실적",
        "누계달성율",
        "연도"
    FROM GN_DW_POC.RAW.FACT_RETRANSMIT_MONTHLY_DEV
)
SELECT * FROM drtv
UNION ALL
SELECT * FROM digital
UNION ALL
SELECT * FROM retransmit

In [ ]:
%%sql -r v2_result
-- ② V_BUDGET_EFFICIENCY: 집행예산 및 개발효율
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_BUDGET_EFFICIENCY AS
WITH drtv AS (
    SELECT
        'DRTV' AS "매체구분",
        "구분" AS "기준일",
        "월 구분" AS "월",
        "월별 예산",
        "집행 예산",
        "누계 집행 예산",
        "누계집행율",
        "연 광고예산",
        "월별 실적" AS "개발건수",
        CASE WHEN "월별 실적" > 0 THEN "집행 예산" / "월별 실적" ELSE NULL END AS "건당비용(CPA)",
        "해당연도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_DRTV_MONTHLY_DEV
),
digital AS (
    SELECT
        '디지털' AS "매체구분",
        "날짜" AS "기준일",
        "월",
        "월별 편성예산(원)" AS "월별 예산",
        "월별 집행예산(원)" AS "집행 예산",
        NULL AS "누계 집행 예산",
        NULL AS "누계집행율",
        "연 광고 예산(원)" AS "연 광고예산",
        "월별 개발실적(건)" AS "개발건수",
        CASE WHEN "월별 개발실적(건)" > 0 THEN "월별 집행예산(원)" / "월별 개발실적(건)" ELSE NULL END AS "건당비용(CPA)",
        "연도"
    FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV
),
retransmit AS (
    SELECT
        '재송출' AS "매체구분",
        "구분" AS "기준일",
        "월 구분" AS "월",
        "월별 예산",
        "집행 예산",
        "누계 집행 예산",
        "누계집행율",
        "연 광고예산",
        "월별 실적" AS "개발건수",
        CASE WHEN "월별 실적" > 0 THEN "집행 예산" / "월별 실적" ELSE NULL END AS "건당비용(CPA)",
        "연도"
    FROM GN_DW_POC.RAW.FACT_RETRANSMIT_MONTHLY_DEV
)
SELECT * FROM drtv
UNION ALL
SELECT * FROM digital
UNION ALL
SELECT * FROM retransmit

In [ ]:
%%sql -r v3a_result
-- ③-a V_MEDIA_EFFICIENCY_DETAIL: 매체별 개발효율 상세
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_MEDIA_EFFICIENCY_DETAIL AS
WITH drtv AS (
    SELECT
        'DRTV' AS "매체구분",
        b."방송일자" AS "날짜",
        b."주차",
        b."요일",
        b."채널" AS "매체",
        b."채널사 유형" AS "매체유형",
        'TV광고' AS "광고유형",
        b."CRM(세부캠페인)명칭" AS "캠페인명",
        c."상위캠페인명",
        c."브랜드명",
        TRY_TO_NUMBER(b."횟수") AS "횟수",
        TRY_TO_NUMBER(b."실구매광고비(원)") AS "광고비",
        TRY_TO_NUMBER(b."인입콜") AS "인입콜",
        TRY_TO_DOUBLE(b."CPC") AS "CPC",
        TRY_TO_DOUBLE(b."광고시청률") AS "광고시청률",
        b."방송월",
        b."해당연도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_DRTV_BROADCAST_EFF b
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
        ON b."CRM(세부캠페인)명칭" = c."세부캠페인명"
),
digital AS (
    SELECT
        '디지털' AS "매체구분",
        d."날짜",
        d."주차",
        d."요일",
        d."매체",
        d."기기" AS "매체유형",
        d."광고유형",
        d."캠페인명",
        d."상위캠페인명",
        NULL AS "브랜드명",
        NULL AS "횟수",
        d."GA 광고비" AS "광고비",
        NULL AS "인입콜",
        CASE WHEN d."클릭수" > 0 THEN d."GA 광고비" / d."클릭수" ELSE NULL END AS "CPC",
        NULL AS "광고시청률",
        d."월" AS "방송월",
        d."연도"
    FROM GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL d
),
retransmit AS (
    SELECT
        '재송출' AS "매체구분",
        r."날짜",
        r."주차",
        r."요일",
        r."방송사" AS "매체",
        r."방송사 유형" AS "매체유형",
        r."방송 대분류" AS "광고유형",
        r."방송명" AS "캠페인명",
        r."상위캠페인명",
        NULL AS "브랜드명",
        r."횟수",
        r."방송편성비" AS "광고비",
        r."인입콜",
        CASE WHEN r."인입콜" > 0 THEN r."방송편성비" / r."인입콜" ELSE NULL END AS "CPC",
        NULL AS "광고시청률",
        r."방송월",
        r."년도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_RETRANSMIT_BROADCAST_CONV r
)
SELECT * FROM drtv
UNION ALL
SELECT * FROM digital
UNION ALL
SELECT * FROM retransmit

In [ ]:
%%sql -r v3b_result
-- ③-b V_TIME_SLOT_EFFICIENCY: 시간대별 효율 (DRTV, 재송출만)
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_TIME_SLOT_EFFICIENCY AS
WITH drtv AS (
    SELECT
        'DRTV' AS "매체구분",
        "방송일자" AS "날짜",
        "시간대",
        "주중/토/일" AS "주중토일",
        "채널" AS "매체",
        "편성명" AS "프로그램",
        TRY_TO_NUMBER("횟수") AS "횟수",
        TRY_TO_NUMBER("실구매광고비(원)") AS "광고비",
        TRY_TO_NUMBER("인입콜") AS "인입콜",
        TRY_TO_DOUBLE("광고시청률") AS "광고시청률",
        "방송월",
        "해당연도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_DRTV_BROADCAST_EFF
),
retransmit AS (
    SELECT
        '재송출' AS "매체구분",
        "날짜",
        "시간대 구분" AS "시간대",
        NULL AS "주중토일",
        "방송사" AS "매체",
        "방송명" AS "프로그램",
        "횟수",
        "방송편성비" AS "광고비",
        "인입콜",
        NULL AS "광고시청률",
        "방송월",
        "년도" AS "연도"
    FROM GN_DW_POC.RAW.FACT_RETRANSMIT_BROADCAST_CONV
)
SELECT * FROM drtv
UNION ALL
SELECT * FROM retransmit

In [ ]:
%%sql -r v3c_result
-- ③-c V_DRTV_SPOT_EFFICIENCY: DRTV CM위치별, RT유형별 효율
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_DRTV_SPOT_EFFICIENCY AS
SELECT
    "CM위치",
    "Spot Type",
    "채널사 유형",
    "방송월",
    "주차",
    "해당연도" AS "연도",
    COUNT(*) AS "편성수",
    SUM(TRY_TO_NUMBER("횟수")) AS "총횟수",
    SUM(TRY_TO_NUMBER("실구매광고비(원)")) AS "총광고비",
    SUM(TRY_TO_NUMBER("인입콜")) AS "총인입콜",
    AVG(TRY_TO_DOUBLE("광고시청률")) AS "평균시청률",
    CASE WHEN SUM(TRY_TO_NUMBER("인입콜")) > 0
         THEN SUM(TRY_TO_NUMBER("실구매광고비(원)")) / SUM(TRY_TO_NUMBER("인입콜"))
         ELSE NULL END AS "CPC"
FROM GN_DW_POC.RAW.FACT_DRTV_BROADCAST_EFF
GROUP BY "CM위치", "Spot Type", "채널사 유형", "방송월", "주차", "해당연도"

In [ ]:
%%sql -r v4_result
-- ④ V_CONVERTED_MEMBER_PROFILE: 전환회원 특성
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CONVERTED_MEMBER_PROFILE AS
SELECT
    d."개발구분(후원중단/재후원/신규/증액/감액)" AS "개발구분",
    d."후원사업",
    d."브랜드",
    d."상위캠페인",
    d."(가입)캠페인",
    m."성별",
    m."연령대",
    m."지역",
    c."상위캠페인명",
    c."브랜드명",
    d."금액",
    d."신청일",
    d."회원번호",
    d."후원번호",
    d."후원사업번호"
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
LEFT JOIN GN_DW_POC.RAW.DIM_MEMBER_ATTRIBUTE m
    ON d."회원번호" = m."회원번호"
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
    ON d."(가입)캠페인" = c."세부캠페인코드"
WHERE d."개발구분(후원중단/재후원/신규/증액/감액)" IN ('신규', '증액', '재후원')

In [ ]:
%%sql -r v5_result
-- ⑤ V_CAMPAIGN_LTV: 캠페인별 LTV
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_LTV AS
SELECT
    n."(가입)캠페인" AS "캠페인코드",
    c."세부캠페인명" AS "캠페인명",
    c."상위캠페인명",
    c."브랜드명",
    COUNT(*) AS "개발건수",
    SUM(n."개발금액") AS "총개발금액",
    SUM(n."25년 총 납입금액") AS "총납입금액",
    AVG(n."25년 총 납입금액") AS "평균납입금액",
    SUM(CASE WHEN n."중단일" IS NOT NULL THEN 1 ELSE 0 END) AS "중단건수",
    AVG(CASE
        WHEN n."중단일" IS NOT NULL AND TRY_TO_DATE(n."중단일") IS NOT NULL
        THEN DATEDIFF('day', TRY_TO_DATE(TO_VARCHAR(n."신청일"), 'YYYYMMDD'), TRY_TO_DATE(n."중단일"))
        ELSE NULL
    END) AS "평균활동일수",
    AVG(n."개발금액") AS "평균개발금액"
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_NEW n
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
    ON n."(가입)캠페인" = c."세부캠페인코드"
GROUP BY n."(가입)캠페인", c."세부캠페인명", c."상위캠페인명", c."브랜드명"

In [ ]:
%%sql -r v6_result
-- ⑥ V_DISCONTINUATION_REPORT: 회원중단보고
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_DISCONTINUATION_REPORT AS
SELECT
    d."회원번호",
    d."회원구분",
    d."납입방식",
    d."후원금액",
    d."가입일",
    d."중단일",
    d."중단사유",
    d."가입캠페인(세부캠페인)" AS "캠페인코드",
    d."브랜드",
    d."상위캠페인",
    d."가입부서(실적부서)" AS "실적부서",
    m."성별",
    m."연령대",
    m."지역",
    c."세부캠페인명" AS "캠페인명",
    c."상위캠페인명",
    c."브랜드명",
    CEIL(DATEDIFF('day', TRY_TO_DATE(TO_VARCHAR(d."중단일"), 'YYYYMMDD'),
                        TRY_TO_DATE(TO_VARCHAR(d."중단일"), 'YYYYMMDD')) / 7.0) AS "중단주차",
    LEFT(TO_VARCHAR(d."중단일"), 6) AS "중단월"
FROM GN_DW_POC.RAW.FACT_DISCONTINUED_MEMBER d
LEFT JOIN GN_DW_POC.RAW.DIM_MEMBER_ATTRIBUTE m
    ON d."회원번호" = m."회원번호"
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
    ON d."가입캠페인(세부캠페인)" = c."세부캠페인코드"

In [ ]:
%%sql -r v7_result
-- ⑦ V_ALIMTALK_EFFECTIVENESS: 서비스효과성분석 (알림톡 한정, 일간)
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_ALIMTALK_EFFECTIVENESS AS
WITH alimtalk_sends AS (
    SELECT
        "회원번호",
        "발송일시",
        "발송구분(대)",
        "발송구분(중)",
        "발송구분(소)",
        "제목",
        "발송상태",
        "성공률(%)"
    FROM GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND
    WHERE "발송구분(대)" = '알림톡' OR "발송구분(중)" LIKE '%알림톡%'
),
marketing_send AS (
    SELECT
        "회원번호",
        "발송일시",
        "발송구분(대)",
        "발송구분(중)",
        "발송구분(소)",
        "브랜드",
        "상위캠페인"
    FROM GN_DW_POC.RAW.FACT_MARKETING_SEND_NEW
)
SELECT
    a."회원번호",
    a."발송일시",
    a."발송구분(대)",
    a."발송구분(중)",
    a."발송구분(소)",
    a."제목",
    a."발송상태",
    a."성공률(%)",
    ms."브랜드",
    ms."상위캠페인",
    CASE WHEN d."회원번호" IS NOT NULL THEN 'Y' ELSE 'N' END AS "전환여부",
    d."개발구분(후원중단/재후원/신규/증액/감액)" AS "전환유형",
    d."금액" AS "전환금액"
FROM alimtalk_sends a
LEFT JOIN marketing_send ms
    ON a."회원번호" = ms."회원번호"
    AND a."발송일시" = ms."발송일시"
LEFT JOIN GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    ON a."회원번호" = d."회원번호"
    AND d."개발구분(후원중단/재후원/신규/증액/감액)" IN ('신규', '증액', '재후원')

In [ ]:
%%sql -r v8_result
-- ⑧ V_APP_ENGAGEMENT: 앱참여수치 (월간)
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_APP_ENGAGEMENT AS
SELECT
    "페이지경로",
    "이벤트이름",
    "회원ID",
    TRY_TO_NUMBER("세션수") AS "세션수",
    TRY_TO_NUMBER("페이지뷰") AS "페이지뷰",
    TRY_TO_NUMBER("활성사용자수") AS "활성사용자수",
    TRY_TO_NUMBER("방문수") AS "방문수",
    TRY_TO_NUMBER("이벤트수") AS "이벤트수"
FROM GN_DW_POC.RAW.FACT_GA_VISITS_APP

In [ ]:
%%sql -r v9_result
-- ⑨ V_CAMPAIGN_FEE_FORECAST: 캠페인별 예상 회비 예측 베이스
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_FEE_FORECAST AS
WITH survival AS (
    SELECT
        n."(가입)캠페인" AS "캠페인코드",
        c."세부캠페인명" AS "캠페인명",
        c."상위캠페인명",
        COUNT(*) AS "총개발건수",
        SUM(CASE WHEN n."중단일" IS NOT NULL THEN 1 ELSE 0 END) AS "중단건수",
        1.0 - (SUM(CASE WHEN n."중단일" IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0)) AS "유지율",
        AVG(n."개발금액") AS "평균개발금액",
        AVG(CASE
            WHEN n."중단일" IS NOT NULL AND TRY_TO_DATE(n."중단일") IS NOT NULL
            THEN DATEDIFF('month', TRY_TO_DATE(TO_VARCHAR(n."신청일"), 'YYYYMMDD'), TRY_TO_DATE(n."중단일"))
            ELSE NULL
        END) AS "평균활동개월수",
        AVG(n."25년 총 납입금액") AS "평균납입금액"
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_NEW n
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
        ON n."(가입)캠페인" = c."세부캠페인코드"
    GROUP BY n."(가입)캠페인", c."세부캠페인명", c."상위캠페인명"
)
SELECT
    s.*,
    dc."총중단건수",
    dc."평균중단활동개월수"
FROM survival s
LEFT JOIN (
    SELECT
        "가입캠페인(세부캠페인)" AS "캠페인코드",
        COUNT(*) AS "총중단건수",
        AVG(CASE
            WHEN TRY_TO_DATE(TO_VARCHAR("중단일"), 'YYYYMMDD') IS NOT NULL
                AND TRY_TO_DATE(TO_VARCHAR("가입일"), 'YYYYMMDD') IS NOT NULL
            THEN DATEDIFF('month', TRY_TO_DATE(TO_VARCHAR("가입일"), 'YYYYMMDD'), TRY_TO_DATE(TO_VARCHAR("중단일"), 'YYYYMMDD'))
            ELSE NULL
        END) AS "평균중단활동개월수"
    FROM GN_DW_POC.RAW.FACT_DISCONTINUED_MEMBER
    GROUP BY "가입캠페인(세부캠페인)"
) dc ON s."캠페인코드" = dc."캠페인코드"

In [ ]:
%%sql -r v10_result
-- ⑩ V_CAMPAIGN_DEV_FORECAST: 캠페인별 개발 건수 예측 베이스
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_DEV_FORECAST AS
WITH monthly_dev AS (
    SELECT
        LEFT(TO_VARCHAR("신청일"), 6) AS "신청월",
        "브랜드",
        "상위캠페인",
        "(가입)캠페인" AS "캠페인코드",
        "개발구분(후원중단/재후원/신규/증액/감액)" AS "개발구분",
        COUNT(*) AS "개발건수",
        SUM("금액") AS "총금액"
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL
    WHERE "개발구분(후원중단/재후원/신규/증액/감액)" IN ('신규', '증액', '재후원')
    GROUP BY LEFT(TO_VARCHAR("신청일"), 6), "브랜드", "상위캠페인",
             "(가입)캠페인", "개발구분(후원중단/재후원/신규/증액/감액)"
),
media_trend AS (
    SELECT '1' AS "월", 'DRTV' AS "매체", "월별 실적" AS "매체실적" FROM GN_DW_POC.RAW.FACT_DRTV_MONTHLY_DEV WHERE "월 구분" IS NOT NULL
    UNION ALL
    SELECT "월", '디지털', "월별 개발실적(건)" FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV WHERE "월" IS NOT NULL
    UNION ALL
    SELECT "월 구분", '재송출', "월별 실적" FROM GN_DW_POC.RAW.FACT_RETRANSMIT_MONTHLY_DEV WHERE "월 구분" IS NOT NULL
)
SELECT
    d."신청월",
    d."브랜드",
    d."상위캠페인",
    d."캠페인코드",
    d."개발구분",
    d."개발건수",
    d."총금액",
    c."세부캠페인명" AS "캠페인명",
    c."상위캠페인명",
    c."브랜드명"
FROM monthly_dev d
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c
    ON d."캠페인코드" = c."세부캠페인코드"

In [ ]:
%%sql -r analytics_check
-- 전체 ANALYTICS 뷰 확인
SELECT TABLE_NAME, TABLE_TYPE
FROM GN_DW_POC.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'ANALYTICS'
ORDER BY TABLE_NAME

In [ ]:
%%sql -r dataframe_1


## 데이터 재적재 (2026-04-08)
1. 공통_회원개발이력_2025_new.xlsx → FACT_MEMBER_DEV_ALL + FACT_MEMBER_DEV_NEW
2. 공통_캠페인코드_new.xlsx → DIM_CAMPAIGN_CODE
3. 공통_조직도정보.xlsx → DIM_ORG_CODE (신규)

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE GN_DW_POC').collect()
session.sql('USE SCHEMA RAW').collect()

stage_path = '@GN_DW_POC.RAW.STG_POC_DATA'

import tempfile, os
tmp_dir = tempfile.mkdtemp()

def download_from_stage(filename):
    local_path = os.path.join(tmp_dir, filename)
    session.sql(f"GET '{stage_path}/{filename}' 'file://{tmp_dir}/'").collect()
    return local_path

print('Step 1: Downloading new files...')
f_dev = download_from_stage('공통_회원개발이력_2025_new.xlsx')
f_camp = download_from_stage('공통_캠페인코드_new.xlsx')
f_org = download_from_stage('공통_조직도정보.xlsx')
print('Download complete.')
print(f'Files: {os.listdir(tmp_dir)}')

In [ ]:
import openpyxl

for fname in ['공통_회원개발이력_2025_new.xlsx', '공통_캠페인코드_new.xlsx', '공통_조직도정보.xlsx']:
    fpath = os.path.join(tmp_dir, fname)
    wb = openpyxl.load_workbook(fpath, read_only=True)
    print(f'\n=== {fname} ===')
    print(f'Sheets: {wb.sheetnames}')
    for sname in wb.sheetnames:
        ws = wb[sname]
        rows = list(ws.iter_rows(max_row=2, values_only=True))
        if rows:
            print(f'  [{sname}] Header: {rows[0]}')
            if len(rows) > 1:
                print(f'  [{sname}] Row 1:  {rows[1]}')
    wb.close()

In [ ]:
from snowflake.snowpark.types import StructType, StructField, StringType, LongType, DoubleType

def cs(df, cols):
    for col in cols:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace({'None': None, 'nan': None, 'NaT': None, 'nat': None})
    return df

def clean_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        mask = df[col].isna()
        df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S')
        df.loc[mask, col] = None
    return df

def load_table(session, table_name, df, create_sql):
    session.sql(f'DROP TABLE IF EXISTS {table_name}').collect()
    session.sql(create_sql).collect()
    from snowflake.connector.pandas_tools import write_pandas
    conn = session.connection
    success, nchunks, nrows, _ = write_pandas(conn, df, table_name, database='GN_DW_POC', schema='RAW', quote_identifiers=True)
    print(f'  -> {table_name}: {nrows:,} rows (success={success})')
    return nrows

print('Utility functions loaded.')

In [ ]:
print('=== Loading FACT_MEMBER_DEV_ALL (new) ===')
f_dev = os.path.join(tmp_dir, '공통_회원개발이력_2025_new.xlsx')
df = pd.read_excel(f_dev, sheet_name='25년 전체개발이력(신규, 증액, 감액, 재후원, 중단)')
kcols = ['순번', '법인구분', '신청일', '실적부서', '실적부서명', '브랜드', '홍보방법', '가입경로', '상위캠페인',
         '(가입)세부캠페인', '(가입)세부캠페인코드', '후원사업', '개발구분(후원중단/재후원/신규/증액/감액)', '회원번호', '금액',
         '중단일', '중단사유', '후원번호', '후원사업번호']
df.columns = kcols
str_cols = ['법인구분', '실적부서', '실적부서명', '브랜드', '홍보방법', '가입경로', '상위캠페인',
            '(가입)세부캠페인', '(가입)세부캠페인코드', '후원사업', '개발구분(후원중단/재후원/신규/증액/감액)', '회원번호', '중단사유']
df = cs(df, str_cols)
df = clean_dates(df, ['중단일'])
print(f'  Rows: {len(df):,}, Columns: {len(df.columns)}')

col_defs = ('"순번" NUMBER, "법인구분" VARCHAR, "신청일" NUMBER, "실적부서" VARCHAR, "실적부서명" VARCHAR, '
            '"브랜드" VARCHAR, "홍보방법" VARCHAR, "가입경로" VARCHAR, "상위캠페인" VARCHAR, '
            '"(가입)세부캠페인" VARCHAR, "(가입)세부캠페인코드" VARCHAR, "후원사업" VARCHAR, '
            '"개발구분(후원중단/재후원/신규/증액/감액)" VARCHAR, '
            '"회원번호" VARCHAR, "금액" NUMBER, "중단일" VARCHAR, "중단사유" VARCHAR, '
            '"후원번호" NUMBER, "후원사업번호" NUMBER')
load_table(session, 'FACT_MEMBER_DEV_ALL', df, f'CREATE TABLE FACT_MEMBER_DEV_ALL ({col_defs})')

In [ ]:
print('=== Loading FACT_MEMBER_DEV_NEW (new) ===')
df2 = pd.read_excel(f_dev, sheet_name='25년 신규, 증액, 재후원 개발건에 대한 이력')
kcols2 = ['순번', '법인구분', '신청일', '실적부서', '실적부서명', '브랜드', '홍보방법', '가입경로', '상위캠페인',
          '(가입)세부캠페인', '(가입)세부캠페인코드', '후원사업', '회원번호', '개발금액', '25년 총 납입금액',
          '중단일', '중단사유', '후원번호', '후원사업번호']
df2.columns = kcols2
str_cols2 = ['법인구분', '실적부서', '실적부서명', '브랜드', '홍보방법', '가입경로', '상위캠페인',
             '(가입)세부캠페인', '(가입)세부캠페인코드', '후원사업', '회원번호', '중단사유']
df2 = cs(df2, str_cols2)
df2 = clean_dates(df2, ['중단일'])
print(f'  Rows: {len(df2):,}, Columns: {len(df2.columns)}')

col_defs2 = ('"순번" NUMBER, "법인구분" VARCHAR, "신청일" NUMBER, "실적부서" VARCHAR, "실적부서명" VARCHAR, '
             '"브랜드" VARCHAR, "홍보방법" VARCHAR, "가입경로" VARCHAR, "상위캠페인" VARCHAR, '
             '"(가입)세부캠페인" VARCHAR, "(가입)세부캠페인코드" VARCHAR, "후원사업" VARCHAR, '
             '"회원번호" VARCHAR, "개발금액" NUMBER, "25년 총 납입금액" NUMBER, '
             '"중단일" VARCHAR, "중단사유" VARCHAR, "후원번호" NUMBER, "후원사업번호" NUMBER')
load_table(session, 'FACT_MEMBER_DEV_NEW', df2, f'CREATE TABLE FACT_MEMBER_DEV_NEW ({col_defs2})')

In [ ]:
print('=== Loading DIM_CAMPAIGN_CODE (new) ===')
f_camp = os.path.join(tmp_dir, '공통_캠페인코드_new.xlsx')
df3 = pd.read_excel(f_camp, sheet_name='공통캠페인코드')
kcols3 = ['브랜드코드', '브랜드명', '브랜드사용여부', '브랜드사용부서', '상위캠페인코드', '상위캠페인명',
          '상위캠페인사용여부', '세부캠페인명', '세부캠페인코드', '세부캠페인사용부서(실적부서)',
          '세부캠페인사용여부', '실적부서코드', '세부캠페인시작일']
df3.columns = kcols3
df3 = cs(df3, kcols3)
print(f'  Rows: {len(df3):,}, Columns: {len(df3.columns)}')

col_defs3 = ', '.join([f'"{ c}" VARCHAR' for c in kcols3])
load_table(session, 'DIM_CAMPAIGN_CODE', df3, f'CREATE TABLE DIM_CAMPAIGN_CODE ({col_defs3})')

In [ ]:
print('=== Loading DIM_ORG_CODE (new) ===')
f_org = os.path.join(tmp_dir, '공통_조직도정보.xlsx')
df4 = pd.read_excel(f_org, sheet_name='sheet1')
kcols4 = ['부서코드', '부서명', '부서경로', '상위부서코드']
df4.columns = kcols4
df4 = cs(df4, kcols4)
print(f'  Rows: {len(df4):,}, Columns: {len(df4.columns)}')
print(f'  Sample:')
print(df4.head(10).to_string())

col_defs4 = ', '.join([f'"{c}" VARCHAR' for c in kcols4])
load_table(session, 'DIM_ORG_CODE', df4, f'CREATE TABLE DIM_ORG_CODE ({col_defs4})')

## 데이터 추가 적재 (2026-04-15)
### 대상: 2026-04-14 업로드 파일
- **교체**: 캠페인코드, 회원개발이력, 회원특성 (DROP → 재생성)
- **추가**: DRTV/디지털/재송출 26년, 중단회원 26년 1~3월 (APPEND)
- **신규**: 납입이력(15개월), 일시→정기 매칭, 일시회원 후원이력, 구글/메타 광고

In [ ]:
import pandas as pd
import openpyxl
import warnings
warnings.filterwarnings('ignore')
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE GN_DW_POC').collect()
session.sql('USE SCHEMA RAW').collect()
session.sql('USE WAREHOUSE COMPUTE_WH').collect()

import tempfile, os, glob
tmp_dir = tempfile.mkdtemp()
stage_path = '@GN_DW_POC.RAW.STG_POC_DATA'

def dl(filename):
    session.file.get(f"{stage_path}/{filename}", tmp_dir)
    local_path = os.path.join(tmp_dir, filename)
    if not os.path.exists(local_path):
        files = glob.glob(os.path.join(tmp_dir, '*'))
        if files:
            local_path = files[0]
    return local_path

def cs(df, cols):
    for col in cols:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace({'None': None, 'nan': None, 'NaT': None, 'nat': None})
    return df

def clean_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        mask = df[col].isna()
        df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S')
        df.loc[mask, col] = None
    return df

def load_replace(df, table_name, col_defs):
    session.sql(f'DROP TABLE IF EXISTS GN_DW_POC.RAW.{table_name}').collect()
    session.sql(f'CREATE TABLE GN_DW_POC.RAW.{table_name} ({col_defs})').collect()
    sp_df = session.create_dataframe(df)
    sp_df.write.mode('append').save_as_table(f'GN_DW_POC.RAW.{table_name}')
    cnt = session.sql(f'SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.{table_name}').collect()[0]['CNT']
    print(f"  {table_name}: {cnt:,} rows")
    return cnt

def load_append(df, table_name):
    sp_df = session.create_dataframe(df)
    sp_df.write.mode('append').save_as_table(f'GN_DW_POC.RAW.{table_name}')
    cnt = session.sql(f'SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.{table_name}').collect()[0]['CNT']
    print(f"  {table_name}: {cnt:,} rows (after append)")
    return cnt

print('Setup complete')

In [ ]:
target_files = [
    '추가_마케팅_26년_재송출대행사.xlsx',
    '추가_회원_중단회원목록_202601_202603.xlsx',
    '신규_공통_납입이력_202501.xlsx',
    '신규_공통_일시회원_to_정기회원_매칭테이블.xlsx',
    '신규_공통_후원이력_일시회원.xlsx',
    '신규_마케팅_광고플랫폼취합(구글,메타).xlsx',
]

print('=== Downloading & Inspecting remaining files ===')
for fname in target_files:
    fp = dl_xlsx(fname)
    with open(fp, 'rb') as f:
        magic = f.read(4)
    if magic[:4] == b'\xd0\xcf\x11\xe0':
        print(f'\n--- {fname} --- (OLE2/.xls format - will use existing 25년 pattern)')
        continue
    wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
    print(f'\n--- {fname} ---')
    print(f'  Sheets: {wb.sheetnames}')
    for sname in wb.sheetnames:
        ws = wb[sname]
        rows = list(ws.iter_rows(max_row=3, values_only=True))
        if rows:
            print(f'  [{sname}] cols={len(rows[0])}, Header: {rows[0]}')
            if len(rows) > 1:
                print(f'    Row1: {rows[1]}')
    wb.close()
print('\nInspection complete.')

In [ ]:
first_batch = [
    '교체_공통_캠페인코드_2026.xlsx',
    '교체_공통_회원개발이력_2025_2026.xlsx',
    '교체_공통_회원특성_2026_1.xlsx',
    '교체_공통_회원특성_2026_2.xlsx',
    '추가_마케팅_26년_DRTV대행사.xlsx',
    '추가_마케팅_26년_디지털대행사.xlsx',
]
for fname in first_batch:
    fp = os.path.join(tmp_dir, fname)
    if not os.path.exists(fp):
        fp = dl_xlsx(fname)
    wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
    print(f'\n--- {fname} ---')
    print(f'  Sheets: {wb.sheetnames}')
    for sname in wb.sheetnames:
        ws = wb[sname]
        rows = list(ws.iter_rows(max_row=3, values_only=True))
        if rows:
            print(f'  [{sname}] cols={len(rows[0])}, Header: {rows[0]}')
            if len(rows) > 1:
                print(f'    Row1: {rows[1]}')
    wb.close()

In [ ]:
fp = os.path.join(tmp_dir, '신규_마케팅_광고플랫폼취합(구글,메타).xlsx')
wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
for sname in wb.sheetnames:
    ws = wb[sname]
    print(f'\n=== {sname} ===')
    for i, row in enumerate(ws.iter_rows(max_row=15, values_only=True)):
        non_null = [v for v in row if v is not None]
        if non_null:
            print(f'  Row {i}: {row[:15]}')
wb.close()

In [ ]:
print('=== Phase 1-1: 교체 DIM_CAMPAIGN_CODE ===')
fp = os.path.join(tmp_dir, '교체_공통_캠페인코드_2026.xlsx')
df = pd.read_excel(fp, sheet_name='공통캠페인코드')
kcols = ['브랜드코드', '브랜드명', '브랜드사용여부', '브랜드사용부서', '상위캠페인코드', '상위캠페인명',
         '상위캠페인사용여부', '세부캠페인명', '세부캠페인코드', '세부캠페인사용부서(실적부서)',
         '세부캠페인사용여부', '실적부서코드', '세부캠페인시작일']
df.columns = kcols
df = cs(df, kcols)
print(f'  Rows: {len(df):,}')
col_defs = ', '.join([f'"{c}" VARCHAR' for c in kcols])
load_replace(df, 'DIM_CAMPAIGN_CODE', col_defs)

In [ ]:
print('=== Phase 1-2: 교체 FACT_MEMBER_DEV_ALL ===')
fp = os.path.join(tmp_dir, '교체_공통_회원개발이력_2025_2026.xlsx')
df = pd.read_excel(fp, sheet_name='Sheet1')
kcols = ['법인구분', '후원신청일', '실적부서코드', '실적부서', '브랜드ID', '브랜드', '홍보방법', '가입경로',
         '상위캠페인코드', '상위캠페인', '세부캠페인코드', '세부캠페인', '후원사업ID', '후원사업',
         '개발구분', '회원번호', '금액', 'SPNSR_NO', 'SPNSR_BSNS_NO']
df.columns = kcols
str_cols = ['법인구분', '실적부서코드', '실적부서', '브랜드', '홍보방법', '가입경로',
            '상위캠페인코드', '상위캠페인', '세부캠페인코드', '세부캠페인', '후원사업',
            '개발구분', '회원번호']
df = cs(df, str_cols)
for nc in ['후원신청일', '브랜드ID', '후원사업ID', '금액', 'SPNSR_NO', 'SPNSR_BSNS_NO']:
    df[nc] = pd.to_numeric(df[nc], errors='coerce')
print(f'  Rows: {len(df):,}')

col_defs = ('"법인구분" VARCHAR, "후원신청일" NUMBER, "실적부서코드" VARCHAR, "실적부서" VARCHAR, '
            '"브랜드ID" NUMBER, "브랜드" VARCHAR, "홍보방법" VARCHAR, "가입경로" VARCHAR, '
            '"상위캠페인코드" VARCHAR, "상위캠페인" VARCHAR, "세부캠페인코드" VARCHAR, "세부캠페인" VARCHAR, '
            '"후원사업ID" NUMBER, "후원사업" VARCHAR, "개발구분" VARCHAR, "회원번호" VARCHAR, '
            '"금액" NUMBER, "SPNSR_NO" NUMBER, "SPNSR_BSNS_NO" NUMBER')
load_replace(df, 'FACT_MEMBER_DEV_ALL', col_defs)

In [ ]:
print('=== Phase 1-3: 교체 DIM_MEMBER_ATTRIBUTE ===')
fp1 = os.path.join(tmp_dir, '교체_공통_회원특성_2026_1.xlsx')
fp2 = os.path.join(tmp_dir, '교체_공통_회원특성_2026_2.xlsx')
df1 = pd.read_excel(fp1, sheet_name='회원특성')
df2 = pd.read_excel(fp2, sheet_name='회원특성')
kcols = ['회원번호', '성별', '연령대', '지역']
df1.columns = kcols
df2.columns = kcols
df = pd.concat([df1, df2], ignore_index=True)
df = cs(df, kcols)
print(f'  File1: {len(df1):,}, File2: {len(df2):,}, Total: {len(df):,}')
col_defs = ', '.join([f'"{c}" VARCHAR' for c in kcols])
load_replace(df, 'DIM_MEMBER_ATTRIBUTE', col_defs)

In [ ]:
print('=== 교체 파일에 FACT_MEMBER_DEV_NEW 시트 없음 ===')
print('새 파일(교체_공통_회원개발이력_2025_2026.xlsx)은 Sheet1 하나만 존재')
print('기존 FACT_MEMBER_DEV_NEW 테이블은 삭제하고, 개발구분 필터로 대체')
session.sql('DROP TABLE IF EXISTS GN_DW_POC.RAW.FACT_MEMBER_DEV_NEW').collect()
print('FACT_MEMBER_DEV_NEW dropped.')

In [ ]:
print('=== Phase 2-1: 추가 DRTV tables ===')
fp = os.path.join(tmp_dir, '추가_마케팅_26년_DRTV대행사.xlsx')

print('\n[DRTV 월별 개발 현황]')
df = pd.read_excel(fp, sheet_name='영상광고 월별 개발 현황')
df = df.iloc[:, :16].copy()
kcols = ['구분', '월별 목표', '월별 실적', '달성율', '누계목표', '누계실적', '누계달성율',
         '월별 예산', '집행 예산', '누계집행율', '누계 집행 예산', '연목표', '연 광고예산',
         '해당연도', '예산절차', '월 구분']
df.columns = kcols
df = clean_dates(df, ['구분'])
for c in ['달성율', '누계달성율', '누계집행율']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = cs(df, ['해당연도', '예산절차', '월 구분'])
print(f'  Rows to append: {len(df):,}')
load_append(df, 'FACT_DRTV_MONTHLY_DEV')

print('\n[월별 편성 및 효율]')
df2 = pd.read_excel(fp, sheet_name='월별 편성 및 효율')
kcols2 = ['채널', '요일', '방송일자', '시간대', '주중/토/일', '프로그램 시작시간', '편성명',
          'CM', 'CM위치', '광고 시작시간', '광고시청률', '광고종료시간', 'Spot Type',
          '횟수', '초수', '실구매광고비(원)', '인입콜', 'CPC', '소재',
          'CRM(세부캠페인)명칭', '소재 일관화', '주차', '요일2', '방송월', '채널사 유형', '해당연도']
df2.columns = kcols2
df2 = cs(df2, kcols2)
print(f'  Rows to append: {len(df2):,}')
load_append(df2, 'FACT_DRTV_BROADCAST_EFF')
print('Phase 2-1 complete.')

In [ ]:
print('=== Phase 2-2: 추가 Digital tables ===')
fp = os.path.join(tmp_dir, '추가_마케팅_26년_디지털대행사.xlsx')

print('\n[디지털 월별 개발 현황]')
df = pd.read_excel(fp, sheet_name='디지털 월별 개발 현황')
kcols = ['연도', '예산절차', '월', '날짜', '월별 개발목표(건)', '월별 개발실적(건)',
         '연목표(건)', '월별 편성예산(원)', '월별 집행예산(원)', '연 광고 예산(원)']
df.columns = kcols
df = cs(df, ['연도', '예산절차', '월', '날짜'])
print(f'  Rows to append: {len(df):,}')
load_append(df, 'FACT_DIGITAL_MONTHLY_DEV')

print('\n[2026년 사단법인 광고운영상세]')
df2 = pd.read_excel(fp, sheet_name='2026년 사단법인 광고운영상세')
kcols2 = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
          '날짜', '주차', '일자', '요일', '캠페인명', '소재', '노출수', '클릭수',
          'GA 광고비', 'GA 전환명수', 'GA 개발건수', '상위캠페인명']
df2.columns = kcols2
df2 = clean_dates(df2, ['날짜'])
for nc in ['노출수', '클릭수', 'GA 광고비', 'GA 전환명수', 'GA 개발건수']:
    df2[nc] = pd.to_numeric(df2[nc], errors='coerce')
str_cols = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
            '주차', '일자', '요일', '캠페인명', '소재', '상위캠페인명']
df2 = cs(df2, str_cols)
print(f'  Rows to append: {len(df2):,}')
load_append(df2, 'FACT_DIGITAL_AD_DETAIL')
print('Phase 2-2 complete.')

In [ ]:
print('=== Phase 2-3: 추가 Retransmit tables (DRM 해제 재업로드) ===')
fp = os.path.join(tmp_dir, '추가_마케팅_26년_재송출대행사.xlsx')

print('\n[재송출 월별 개발 현황]')
df = pd.read_excel(fp, sheet_name='재송출 월별 개발 현황')
df = df.iloc[:, :18].copy()
kcols = ['구분', '월별 목표', '월별 실적', '달성율', '누계목표', '누계실적', '누계달성율',
         '월별 예산', '집행 예산', '누계집행율', '누계 집행 예산', '연목표', '연 광고예산',
         '연도', '예산절차', '월 구분', '방송구분', '실적부서']
df.columns = kcols
df = clean_dates(df, ['구분'])
for c in ['달성율', '누계달성율', '누계집행율']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = cs(df, ['연도', '예산절차', '월 구분', '방송구분', '실적부서'])
print(f'  Rows to append: {len(df):,}')
load_append(df, 'FACT_RETRANSMIT_MONTHLY_DEV')

print('\n[방송 송출 및 전환현황]')
df2 = pd.read_excel(fp, sheet_name='방송 송출 및 전환현황')
df2 = df2.iloc[:, :20].copy()
kcols2 = ['구분', '년도', '방송월', '방송사', '방송명', '상위캠페인명', '본방송 구분',
          '날짜', '요일', '방송시간', '인입콜', '회원개발(건)', '방송편성비',
          '주차', '횟수', '시간대 구분', '국내/해외 구분', '실적부서', '방송사 유형', '방송 대분류']
df2.columns = kcols2
df2 = clean_dates(df2, ['날짜'])
df2['방송시간'] = df2['방송시간'].apply(lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x) if pd.notna(x) else None)
str_cols = ['구분', '년도', '방송월', '방송사', '방송명', '상위캠페인명', '본방송 구분',
            '요일', '방송시간', '주차', '시간대 구분', '국내/해외 구분', '실적부서',
            '방송사 유형', '방송 대분류']
df2 = cs(df2, str_cols)
print(f'  Rows to append: {len(df2):,}')
load_append(df2, 'FACT_RETRANSMIT_BROADCAST_CONV')
print('\nPhase 2-3 complete.')

In [ ]:
print('=== Phase 2-4: 추가 FACT_DISCONTINUED_MEMBER ===')
fp = os.path.join(tmp_dir, '추가_회원_중단회원목록_202601_202603.xlsx')
df = pd.read_excel(fp, sheet_name='Sheet')
kcols = ['순번', '법인구분', '회원번호', '회원구분', '납입방식', '후원금액', '가입일', '중단일',
         '중단사유', '브랜드', '상위캠페인', '가입캠페인(세부캠페인)', '가입부서(실적부서)']
df.columns = kcols

for dc in ['가입일', '중단일']:
    df[dc] = pd.to_datetime(df[dc], errors='coerce')
    df[dc] = df[dc].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) else None)

str_cols = ['법인구분', '회원번호', '회원구분', '납입방식', '중단사유',
            '가입캠페인(세부캠페인)', '브랜드', '상위캠페인', '가입부서(실적부서)']
df = cs(df, str_cols)
print(f'  Rows to append: {len(df):,}')

reorder = ['순번', '법인구분', '회원번호', '회원구분', '납입방식', '후원금액', '가입일', '중단일',
           '중단사유', '가입캠페인(세부캠페인)', '브랜드', '상위캠페인', '가입부서(실적부서)']
df = df[reorder]
load_append(df, 'FACT_DISCONTINUED_MEMBER')
print('Phase 2-4 complete.')

In [ ]:
print('=== Phase 3-1: 신규 FACT_PAYMENT_HISTORY (15 files) ===')
kcols = ['회원번호', 'SPNSR_NO', 'SPNSR_BSNS_NO', '회비청구월', '청구금액', '납입금액', '납입일', '납입구분', '비고']
col_defs = ('"회원번호" VARCHAR, "SPNSR_NO" NUMBER, "SPNSR_BSNS_NO" NUMBER, '
            '"회비청구월" NUMBER, "청구금액" NUMBER, "납입금액" NUMBER, '
            '"납입일" VARCHAR, "납입구분" VARCHAR, "비고" VARCHAR')
session.sql(f'DROP TABLE IF EXISTS GN_DW_POC.RAW.FACT_PAYMENT_HISTORY').collect()
session.sql(f'CREATE TABLE GN_DW_POC.RAW.FACT_PAYMENT_HISTORY ({col_defs})').collect()

months = [
    '202501', '202502', '202503', '202504', '202505', '202506',
    '202507', '202508', '202509', '202510', '202511', '202512',
    '202601', '202602', '202603'
]
total = 0
for m in months:
    fname = f'신규_공통_납입이력_{m}.xlsx'
    print(f'  {fname}...', end='', flush=True)
    fp = dl_xlsx(fname)
    df = pd.read_excel(fp, sheet_name='Sheet1')
    df = df.iloc[:, :9].copy()
    df.columns = kcols
    df = cs(df, ['회원번호', '납입구분', '비고'])
    df = clean_dates(df, ['납입일'])
    for nc in ['SPNSR_NO', 'SPNSR_BSNS_NO', '회비청구월', '청구금액', '납입금액']:
        df[nc] = pd.to_numeric(df[nc], errors='coerce')
    df['비고'] = df['비고'].replace({'NULL': None})
    sp_df = session.create_dataframe(df)
    sp_df.write.mode('append').save_as_table('GN_DW_POC.RAW.FACT_PAYMENT_HISTORY')
    total += len(df)
    print(f' {len(df):,} rows (누적: {total:,})')

cnt = session.sql('SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_PAYMENT_HISTORY').collect()[0]['CNT']
print(f'\nFACT_PAYMENT_HISTORY total: {cnt:,} rows')

In [ ]:
print('=== Phase 3-2: 신규 DIM_TEMP_TO_REGULAR_MATCH ===')
fp = dl_xlsx('신규_공통_일시회원_to_정기회원_매칭테이블.xlsx')
df = pd.read_excel(fp, sheet_name='Sheet1')
df = df.iloc[:, :3].copy()
kcols = ['회원번호(정기)', '회원번호(일시)', '전환일']
df.columns = kcols
df = cs(df, ['회원번호(정기)', '회원번호(일시)'])
df = clean_dates(df, ['전환일'])
print(f'  Rows: {len(df):,}')
col_defs = '"회원번호(정기)" VARCHAR, "회원번호(일시)" VARCHAR, "전환일" VARCHAR'
load_replace(df, 'DIM_TEMP_TO_REGULAR_MATCH', col_defs)

In [ ]:
print('=== Phase 3-3: 신규 FACT_TEMP_MEMBER_DONATION ===')
fp = dl_xlsx('신규_공통_후원이력_일시회원.xlsx')
df = pd.read_excel(fp, sheet_name='Sheet1')
df = df.iloc[:, :7].copy()
kcols = ['일시회원번호', '후원금액', '후원일', '실적부서코드', '실적부서명', '세부캠페인코드', '세부캠페인명']
df.columns = kcols
df = cs(df, ['일시회원번호', '실적부서코드', '실적부서명', '세부캠페인명'])
df['세부캠페인코드'] = pd.to_numeric(df['세부캠페인코드'], errors='coerce')
df['후원금액'] = pd.to_numeric(df['후원금액'], errors='coerce')
df = clean_dates(df, ['후원일'])
print(f'  Rows: {len(df):,}')
col_defs = ('"일시회원번호" VARCHAR, "후원금액" NUMBER, "후원일" VARCHAR, '
            '"실적부서코드" VARCHAR, "실적부서명" VARCHAR, "세부캠페인코드" NUMBER, "세부캠페인명" VARCHAR')
load_replace(df, 'FACT_TEMP_MEMBER_DONATION', col_defs)

In [ ]:
print('=== Phase 3-4: 신규 FACT_AD_PLATFORM_GOOGLE_META ===')
fp = dl_xlsx('신규_마케팅_광고플랫폼취합(구글,메타).xlsx')

print('\n[Sheet 1: GA - 잠재고객+캠페인+회원번호]')
df_ga = pd.read_excel(fp, sheet_name='GA(잠재고객+캠페인+회원번호)', skiprows=5)
df_ga.columns = ['날짜', '잠재고객이름', '세션캠페인', '회원번호', '세션수', '활성사용자']
df_ga = cs(df_ga, ['잠재고객이름', '세션캠페인', '회원번호'])
df_ga['날짜'] = pd.to_numeric(df_ga['날짜'], errors='coerce')
for nc in ['세션수', '활성사용자']:
    df_ga[nc] = pd.to_numeric(df_ga[nc], errors='coerce')
df_ga['플랫폼'] = 'GA'
df_ga['시트구분'] = 'GA_잠재고객'
print(f'  Rows: {len(df_ga):,}')

print('\n[Sheet 2: Google Ads Demandgen]')
df_dg = pd.read_excel(fp, sheet_name='Google Ads_Demandgen', skiprows=3)
df_dg_cols = list(df_dg.columns)
print(f'  Original cols: {df_dg_cols}')
print(f'  Rows: {len(df_dg):,}')

print('\n[Sheet 3: Google Ads PMAX]')
df_pm = pd.read_excel(fp, sheet_name='Google Ads_PMAX', skiprows=3)
df_pm_cols = list(df_pm.columns)
print(f'  Original cols: {df_pm_cols}')
print(f'  Rows: {len(df_pm):,}')

print('\n[Sheet 4: 메타]')
df_meta = pd.read_excel(fp, sheet_name='메타')
print(f'  Cols: {list(df_meta.columns)}')
print(f'  Rows: {len(df_meta):,}')
print(f'  Sample: {df_meta.head(2).to_dict()}')

In [ ]:
print('=== Phase 3-4b: Load all Google/Meta ad platform tables ===')
fp = os.path.join(tmp_dir, '신규_마케팅_광고플랫폼취합(구글,메타).xlsx')

print('\n[1/4] FACT_AD_GA_AUDIENCE')
df_ga = pd.read_excel(fp, sheet_name='GA(잠재고객+캠페인+회원번호)', skiprows=5)
df_ga = df_ga.iloc[:, :6].copy()
kcols_ga = ['날짜', '잠재고객이름', '세션캠페인', '회원번호', '세션수', '활성사용자']
df_ga.columns = kcols_ga
df_ga = cs(df_ga, ['잠재고객이름', '세션캠페인', '회원번호'])
for nc in ['날짜', '세션수', '활성사용자']:
    df_ga[nc] = pd.to_numeric(df_ga[nc], errors='coerce')
df_ga = df_ga.dropna(subset=['날짜'])
print(f'  Rows: {len(df_ga):,}')
col_defs = '"날짜" NUMBER, "잠재고객이름" VARCHAR, "세션캠페인" VARCHAR, "회원번호" VARCHAR, "세션수" NUMBER, "활성사용자" NUMBER'
load_replace(df_ga, 'FACT_AD_GA_AUDIENCE', col_defs)

print('\n[2/4] FACT_AD_GOOGLE_DEMANDGEN')
df_dg = pd.read_excel(fp, sheet_name='Google Ads_Demandgen', header=None, skiprows=3)
df_dg = df_dg.iloc[:, :13].copy()
kcols_dg = ['날짜', '캠페인유형', '캠페인이름', '기기', '타겟팅', '방문페이지',
            '노출수', '클릭수', '통화', '비용', '전환수', '전환율', '전환가치']
df_dg.columns = kcols_dg
df_dg = cs(df_dg, ['캠페인유형', '캠페인이름', '기기', '타겟팅', '방문페이지', '통화'])
df_dg = clean_dates(df_dg, ['날짜'])
for nc in ['노출수', '클릭수', '비용', '전환수', '전환율', '전환가치']:
    df_dg[nc] = pd.to_numeric(df_dg[nc], errors='coerce')
df_dg = df_dg.dropna(subset=['캠페인이름'])
print(f'  Rows: {len(df_dg):,}')
col_defs = ('"날짜" VARCHAR, "캠페인유형" VARCHAR, "캠페인이름" VARCHAR, "기기" VARCHAR, '
            '"타겟팅" VARCHAR, "방문페이지" VARCHAR, "노출수" NUMBER, "클릭수" NUMBER, '
            '"통화" VARCHAR, "비용" NUMBER, "전환수" NUMBER, "전환율" FLOAT, "전환가치" FLOAT')
load_replace(df_dg, 'FACT_AD_GOOGLE_DEMANDGEN', col_defs)

print('\n[3/4] FACT_AD_GOOGLE_PMAX')
df_pm = pd.read_excel(fp, sheet_name='Google Ads_PMAX', header=None, skiprows=3)
df_pm = df_pm.iloc[:, :13].copy()
kcols_pm = ['날짜', '캠페인유형', '캠페인이름', '기기', '방문페이지', '최종URL',
            '노출수', '클릭수', '통화', '비용', '전환수', '전환율', '전환가치']
df_pm.columns = kcols_pm
df_pm = cs(df_pm, ['캠페인유형', '캠페인이름', '기기', '방문페이지', '최종URL', '통화'])
df_pm = clean_dates(df_pm, ['날짜'])
for nc in ['노출수', '클릭수', '비용', '전환수', '전환율', '전환가치']:
    df_pm[nc] = pd.to_numeric(df_pm[nc], errors='coerce')
df_pm = df_pm.dropna(subset=['캠페인이름'])
print(f'  Rows: {len(df_pm):,}')
col_defs = ('"날짜" VARCHAR, "캠페인유형" VARCHAR, "캠페인이름" VARCHAR, "기기" VARCHAR, '
            '"방문페이지" VARCHAR, "최종URL" VARCHAR, "노출수" NUMBER, "클릭수" NUMBER, '
            '"통화" VARCHAR, "비용" NUMBER, "전환수" NUMBER, "전환율" FLOAT, "전환가치" FLOAT')
load_replace(df_pm, 'FACT_AD_GOOGLE_PMAX', col_defs)

print('\n[4/4] FACT_AD_META')
df_meta = pd.read_excel(fp, sheet_name='메타')
kcols_meta = ['일', '캠페인이름', '광고세트이름', '광고이름', '노출', '링크클릭',
              '지출금액_KRW', '캠페인예산', '캠페인예산유형', '광고세트예산',
              '광고세트예산유형', '구매', '구매당비용', '보고시작', '보고종료']
df_meta.columns = kcols_meta
df_meta = cs(df_meta, ['일', '캠페인이름', '광고세트이름', '광고이름', '캠페인예산',
                       '캠페인예산유형', '광고세트예산유형', '보고시작', '보고종료'])
for nc in ['노출', '링크클릭', '지출금액_KRW', '광고세트예산', '구매', '구매당비용']:
    df_meta[nc] = pd.to_numeric(df_meta[nc], errors='coerce')
print(f'  Rows: {len(df_meta):,}')
col_defs = ('"일" VARCHAR, "캠페인이름" VARCHAR, "광고세트이름" VARCHAR, "광고이름" VARCHAR, '
            '"노출" NUMBER, "링크클릭" FLOAT, "지출금액_KRW" NUMBER, "캠페인예산" VARCHAR, '
            '"캠페인예산유형" VARCHAR, "광고세트예산" FLOAT, "광고세트예산유형" VARCHAR, '
            '"구매" FLOAT, "구매당비용" FLOAT, "보고시작" VARCHAR, "보고종료" VARCHAR')
load_replace(df_meta, 'FACT_AD_META', col_defs)

print('\nPhase 3-4 complete.')

In [ ]:
%%sql -r raw_verify_20260415
SELECT TABLE_NAME, ROW_COUNT
FROM GN_DW_POC.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'RAW' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME

In [ ]:
%%sql -r v_converted_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CONVERTED_MEMBER_PROFILE AS
SELECT
    d."개발구분" AS "개발구분",
    d."후원사업", d."브랜드", d."상위캠페인",
    d."세부캠페인", d."세부캠페인코드",
    m."성별", m."연령대", m."지역",
    COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
    d."브랜드" AS "브랜드명",
    d."금액", d."후원신청일" AS "신청일",
    LEFT(TO_VARCHAR(d."후원신청일"), 6) AS "신청월",
    d."회원번호", d."SPNSR_NO" AS "후원번호", d."SPNSR_BSNS_NO" AS "후원사업번호",
    d."실적부서코드" AS "실적부서", o."부서명" AS "실적부서명",
    d."홍보방법", d."법인구분", d."가입경로",
    COALESCE(c."세부캠페인명", d."세부캠페인") AS "세부캠페인명"
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
LEFT JOIN GN_DW_POC.RAW.DIM_MEMBER_ATTRIBUTE m ON d."회원번호" = m."회원번호"
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
LEFT JOIN GN_DW_POC.RAW.DIM_ORG_CODE o ON d."실적부서코드" = o."부서코드"
WHERE d."개발구분" IN ('신규', '증액', '재후원')

In [ ]:
%%sql -r v_ltv_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_LTV AS
SELECT
    d."세부캠페인코드" AS "캠페인코드",
    COALESCE(c."세부캠페인명", d."세부캠페인") AS "캠페인명",
    COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
    COALESCE(c."브랜드명", d."브랜드") AS "브랜드명",
    COUNT(*) AS "개발건수",
    SUM(d."금액") AS "총개발금액",
    NULL AS "총납입금액",
    NULL AS "평균납입금액",
    0 AS "중단건수",
    NULL AS "평균활동일수",
    AVG(d."금액") AS "평균개발금액",
    d."세부캠페인코드" AS CAMPAIGN_CODE
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
WHERE d."개발구분" IN ('신규', '증액', '재후원')
GROUP BY d."세부캠페인코드", c."세부캠페인명", c."상위캠페인명", c."브랜드명", d."상위캠페인", d."브랜드", d."세부캠페인"

In [ ]:
%%sql -r v_fee_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_FEE_FORECAST AS
WITH survival AS (
    SELECT d."세부캠페인코드" AS "캠페인코드",
        COALESCE(c."세부캠페인명", d."세부캠페인") AS "캠페인명",
        COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
        COUNT(*) AS "총개발건수",
        0 AS "중단건수",
        1.0 AS "유지율",
        AVG(d."금액") AS "평균개발금액",
        NULL AS "평균활동개월수",
        NULL AS "평균납입금액"
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
    WHERE d."개발구분" IN ('신규', '증액', '재후원')
    GROUP BY d."세부캠페인코드", c."세부캠페인명", c."상위캠페인명", d."상위캠페인", d."세부캠페인"
)
SELECT *, "중단건수" AS "총중단건수", "평균활동개월수" AS "평균중단활동개월수"
FROM survival

In [ ]:
%%sql -r v_dev_forecast_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_DEV_FORECAST AS
SELECT
    LEFT(TO_VARCHAR(d."후원신청일"), 6) AS "신청월",
    d."브랜드",
    d."상위캠페인",
    d."세부캠페인코드" AS "캠페인코드",
    d."개발구분",
    COUNT(*) AS "개발건수",
    SUM(d."금액") AS "총금액",
    COALESCE(c."세부캠페인명", d."세부캠페인") AS "캠페인명",
    COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
    d."브랜드" AS "브랜드명",
    o."부서명" AS "실적부서명",
    d."실적부서코드" AS "실적부서",
    d."후원사업",
    d."홍보방법",
    d."법인구분"
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
LEFT JOIN GN_DW_POC.RAW.DIM_ORG_CODE o ON d."실적부서코드" = o."부서코드"
WHERE d."개발구분" IN ('신규', '증액', '재후원')
GROUP BY LEFT(TO_VARCHAR(d."후원신청일"), 6), d."브랜드", d."상위캠페인",
         d."세부캠페인코드", d."개발구분",
         c."세부캠페인명", c."상위캠페인명", d."세부캠페인",
         o."부서명", d."실적부서코드", d."후원사업", d."홍보방법", d."법인구분"

In [ ]:
%%sql -r v_detail_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_MEMBER_DEV_DETAIL AS
SELECT
    LEFT(TO_VARCHAR(d."후원신청일"), 6) AS "신청월",
    d."법인구분", d."실적부서코드" AS "실적부서",
    o."부서명" AS "실적부서명",
    d."브랜드" AS "브랜드명", d."홍보방법", d."가입경로", d."상위캠페인",
    COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
    d."세부캠페인" AS "세부캠페인",
    COALESCE(c."세부캠페인명", d."세부캠페인") AS "세부캠페인명",
    d."세부캠페인코드",
    d."후원사업",
    d."개발구분",
    d."회원번호", d."금액",
    d."SPNSR_NO" AS "후원번호", d."SPNSR_BSNS_NO" AS "후원사업번호",
    m."성별", m."연령대", m."지역",
    d."금액" / 10000.0 AS "개발실적_건",
    d."회원번호" AS MEMBER_ID,
    d."세부캠페인코드" AS CAMPAIGN_CODE,
    COALESCE(c."상위캠페인명", d."상위캠페인") AS PARENT_CAMPAIGN,
    LEFT(TO_VARCHAR(d."후원신청일"), 6) AS APPLY_MONTH,
    d."개발구분" AS DEV_TYPE
FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
LEFT JOIN GN_DW_POC.RAW.DIM_MEMBER_ATTRIBUTE m ON d."회원번호" = m."회원번호"
LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
LEFT JOIN GN_DW_POC.RAW.DIM_ORG_CODE o ON d."실적부서코드" = o."부서코드"

In [ ]:
%%sql -r v_channel_roi_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CHANNEL_ROI AS
WITH channel_cost AS (
    SELECT "매체" AS channel_name, "매체구분", "매체유형" AS channel_type,
        SUM("광고비") AS total_ad_cost, SUM("인입콜") AS total_call, SUM("횟수") AS total_freq,
        COUNT(DISTINCT "상위캠페인명") AS linked_campaigns
    FROM GN_DW_POC.ANALYTICS.V_MEDIA_EFFICIENCY_DETAIL
    WHERE "매체" IS NOT NULL AND "매체" != ''
    GROUP BY "매체", "매체구분", "매체유형"
),
channel_campaigns AS (
    SELECT DISTINCT "매체" AS channel_name, "상위캠페인명"
    FROM GN_DW_POC.ANALYTICS.V_MEDIA_EFFICIENCY_DETAIL WHERE "매체" IS NOT NULL AND "상위캠페인명" IS NOT NULL
),
campaign_dev AS (
    SELECT COALESCE(c."상위캠페인명", d."상위캠페인") AS campaign_name,
        COUNT(*) AS dev_count, SUM(d."금액") / 10000.0 AS dev_performance,
        SUM(CASE WHEN d."개발구분" = '신규' THEN 1 ELSE 0 END) AS new_sponsors
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
    WHERE d."개발구분" IN ('신규', '증액', '재후원')
    GROUP BY COALESCE(c."상위캠페인명", d."상위캠페인")
)
SELECT cc.channel_name, cc."매체구분", cc.channel_type,
    cc.total_ad_cost, cc.total_call, cc.total_freq, cc.linked_campaigns,
    SUM(cd.dev_count) AS total_dev, SUM(cd.dev_performance) AS total_dev_perf,
    SUM(cd.new_sponsors) AS total_new_sponsors,
    CASE WHEN SUM(cd.dev_count) > 0 THEN cc.total_ad_cost / SUM(cd.dev_count) ELSE NULL END AS cpa
FROM channel_cost cc
LEFT JOIN channel_campaigns link ON cc.channel_name = link.channel_name
LEFT JOIN campaign_dev cd ON link."상위캠페인명" = cd.campaign_name
GROUP BY cc.channel_name, cc."매체구분", cc.channel_type, cc.total_ad_cost, cc.total_call, cc.total_freq, cc.linked_campaigns

In [ ]:
%%sql -r v_campaign_roi_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_CAMPAIGN_ROI AS
WITH ad_cost_by_campaign AS (
    SELECT "상위캠페인명", SUM("광고비") AS total_ad_cost
    FROM GN_DW_POC.ANALYTICS.V_MEDIA_EFFICIENCY_DETAIL
    WHERE "상위캠페인명" IS NOT NULL AND "상위캠페인명" != ''
    GROUP BY "상위캠페인명"
),
dev_data AS (
    SELECT COALESCE(c."상위캠페인명", d."상위캠페인") AS campaign_name,
        COUNT(*) AS total_dev_raw, SUM(d."금액") / 10000.0 AS dev_performance,
        SUM(CASE WHEN d."개발구분" = '신규' THEN 1 ELSE 0 END) AS new_sponsors,
        SUM(d."금액") AS total_pledge_amount
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
    WHERE d."개발구분" IN ('신규', '증액', '재후원')
    GROUP BY COALESCE(c."상위캠페인명", d."상위캠페인")
)
SELECT
    COALESCE(a."상위캠페인명", d.campaign_name) AS "캠페인명",
    COALESCE(a.total_ad_cost, 0) AS "총광고비",
    COALESCE(d.total_dev_raw, 0) AS "총개발건수",
    COALESCE(d.dev_performance, 0) AS "총개발실적",
    COALESCE(d.new_sponsors, 0) AS "신규후원자수",
    COALESCE(d.total_pledge_amount, 0) AS "총약정금액",
    CASE WHEN d.total_dev_raw > 0 THEN a.total_ad_cost / d.total_dev_raw ELSE NULL END AS "CPA",
    CASE WHEN a.total_ad_cost > 0 THEN d.total_pledge_amount / a.total_ad_cost ELSE NULL END AS "ROI"
FROM ad_cost_by_campaign a
FULL OUTER JOIN dev_data d ON a."상위캠페인명" = d.campaign_name
WHERE COALESCE(a."상위캠페인명", d.campaign_name) IS NOT NULL

In [ ]:
%%sql -r v_send_conv_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_SEND_CONVERSION_ANALYSIS AS
WITH sends AS (
    SELECT
        CASE
            WHEN "제목" LIKE '%알림톡%' OR "발송구분(소)" LIKE '%알림%' THEN '알림톡'
            WHEN "발송구분(대)" IN ('참여') THEN '마케팅메일'
            ELSE '문자/SMS'
        END AS send_type,
        "발송구분(대)",
        "회원번호",
        TRY_TO_TIMESTAMP("발송일시") AS send_ts
    FROM GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND
    WHERE "회원번호" IS NOT NULL AND "발송일시" IS NOT NULL
),
conversions AS (
    SELECT
        "회원번호",
        TRY_TO_DATE(TO_VARCHAR("후원신청일"), 'YYYYMMDD') AS conv_date,
        "금액",
        "개발구분" AS dev_type
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL
    WHERE "개발구분" IN ('신규', '증액', '재후원')
),
matched AS (
    SELECT
        s.send_type, s."발송구분(대)", s."회원번호", s.send_ts,
        c.conv_date, c."금액",
        DATEDIFF('day', s.send_ts::DATE, c.conv_date) AS days_to_conv
    FROM sends s
    INNER JOIN conversions c ON s."회원번호" = c."회원번호"
    WHERE c.conv_date >= s.send_ts::DATE AND c.conv_date <= DATEADD('day', 90, s.send_ts::DATE)
)
SELECT
    send_type AS "발송유형", "발송구분(대)" AS "발송구분_대",
    COUNT(*) AS "전환건수", AVG(days_to_conv) AS "평균전환소요일",
    AVG("금액") AS "평균전환금액",
    COUNT(*) * 1.0 / NULLIF((SELECT COUNT(DISTINCT "회원번호") FROM sends s2 WHERE s2.send_type = matched.send_type), 0) AS "전환율",
    (SELECT COUNT(DISTINCT "회원번호") FROM sends s3 WHERE s3.send_type = matched.send_type) AS "총발송건수"
FROM matched
GROUP BY send_type, "발송구분(대)"

In [ ]:
%%sql -r v_alimtalk_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_ALIMTALK_EFFECTIVENESS AS
WITH alimtalk_sends AS (
    SELECT "회원번호", "발송일시", "발송구분(대)", "발송구분(중)", "발송구분(소)",
        "제목", "발송상태", "성공률(%)"
    FROM GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND
    WHERE "제목" LIKE '%알림톡%' OR "발송구분(소)" LIKE '%알림%'
),
marketing_send AS (
    SELECT "회원번호", "발송일시", "발송구분(대)", "발송구분(중)", "발송구분(소)", "브랜드", "상위캠페인"
    FROM GN_DW_POC.RAW.FACT_MARKETING_SEND_NEW
)
SELECT
    a."회원번호", a."발송일시", a."발송구분(대)", a."발송구분(중)", a."발송구분(소)",
    a."제목", a."발송상태", a."성공률(%)",
    ms."브랜드", ms."상위캠페인",
    CASE WHEN d."회원번호" IS NOT NULL THEN 'Y' ELSE 'N' END AS "전환여부",
    d."개발구분" AS "전환유형",
    d."금액" AS "전환금액"
FROM alimtalk_sends a
LEFT JOIN marketing_send ms ON a."회원번호" = ms."회원번호" AND a."발송일시" = ms."발송일시"
LEFT JOIN GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d ON a."회원번호" = d."회원번호"
    AND d."개발구분" IN ('신규', '증액', '재후원')

In [ ]:
%%sql -r v_alimtalk_cross_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_ALIMTALK_INCREASE_CROSS AS
WITH alimtalk_members AS (
    SELECT DISTINCT "회원번호", LEFT("발송일시", 7) AS send_month
    FROM GN_DW_POC.RAW.FACT_SMS_ALIMTALK_SEND
    WHERE ("제목" LIKE '%알림톡%' OR "발송구분(소)" LIKE '%알림%')
      AND "발송상태" = '발송완료' AND "회원번호" IS NOT NULL
),
increase_dev AS (
    SELECT d."회원번호",
        LEFT(TO_VARCHAR(d."후원신청일"), 6) AS dev_month,
        LEFT(TO_VARCHAR(d."후원신청일"), 4) AS dev_year,
        COALESCE(c."상위캠페인명", d."상위캠페인") AS "상위캠페인명",
        COALESCE(c."세부캠페인명", d."세부캠페인") AS "세부캠페인명",
        d."금액", d."개발구분"
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
    WHERE d."개발구분" = '증액'
)
SELECT
    i."회원번호", i.dev_month AS "개발월", i.dev_year AS "개발연도",
    i."상위캠페인명", i."세부캠페인명", i."금액", i."개발구분",
    CASE WHEN a."회원번호" IS NOT NULL THEN 'Y' ELSE 'N' END AS "알림톡수신여부"
FROM increase_dev i
LEFT JOIN alimtalk_members a ON i."회원번호" = a."회원번호"

In [ ]:
%%sql -r v_loyal_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_LOYAL_MEMBER_ANALYSIS AS
WITH member_history AS (
    SELECT d."회원번호",
        COALESCE(c."상위캠페인명", d."상위캠페인") AS parent_campaign,
        COALESCE(c."세부캠페인명", d."세부캠페인") AS sub_campaign,
        COALESCE(c."브랜드명", d."브랜드") AS brand,
        d."후원사업", d."후원신청일" AS "신청일", d."금액",
        d."개발구분" AS dev_type
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
),
member_first AS (
    SELECT "회원번호", MIN(parent_campaign) AS parent_campaign, MIN(sub_campaign) AS sub_campaign,
        MIN(brand) AS brand, MIN("후원사업") AS sponsorship, MIN("신청일") AS first_apply
    FROM member_history WHERE dev_type IN ('신규', '증액', '재후원')
    GROUP BY "회원번호"
),
member_agg AS (
    SELECT "회원번호", COUNT(*) AS total_dev, SUM("금액") AS total_pledge,
        SUM(CASE WHEN dev_type = '증액' THEN 1 ELSE 0 END) AS increase_count,
        SUM(CASE WHEN dev_type = '증액' THEN "금액" ELSE 0 END) AS increase_amount,
        SUM(CASE WHEN dev_type = '재후원' THEN 1 ELSE 0 END) AS re_sponsor_count
    FROM member_history
    GROUP BY "회원번호"
)
SELECT f."회원번호", f.parent_campaign AS "상위캠페인명", f.sub_campaign AS "세부캠페인명",
    f.brand AS "브랜드명", f.sponsorship AS "후원사업",
    f.first_apply AS "최초신청일",
    a.total_dev AS "총개발횟수", a.total_pledge AS "총약정금액",
    a.increase_count AS "증액횟수", a.increase_amount AS "증액금액",
    a.re_sponsor_count AS "재후원횟수",
    m."성별", m."연령대", m."지역"
FROM member_first f
JOIN member_agg a ON f."회원번호" = a."회원번호"
LEFT JOIN GN_DW_POC.RAW.DIM_MEMBER_ATTRIBUTE m ON f."회원번호" = m."회원번호"
WHERE a.total_dev >= 2

In [ ]:
%%sql -r v_retention_fix
CREATE OR REPLACE VIEW GN_DW_POC.ANALYTICS.V_RETENTION_BY_PERIOD AS
WITH member_data AS (
    SELECT COALESCE(c."상위캠페인명", d."상위캠페인") AS parent_campaign,
        COALESCE(c."세부캠페인명", d."세부캠페인") AS sub_campaign,
        COALESCE(c."브랜드명", d."브랜드") AS brand,
        d."회원번호", d."후원신청일" AS "신청일", d."금액",
        DATEDIFF('month', TRY_TO_DATE(TO_VARCHAR(d."후원신청일"), 'YYYYMMDD'), CURRENT_DATE()) AS active_months
    FROM GN_DW_POC.RAW.FACT_MEMBER_DEV_ALL d
    LEFT JOIN GN_DW_POC.RAW.DIM_CAMPAIGN_CODE c ON d."세부캠페인코드" = c."세부캠페인코드"
    WHERE d."개발구분" IN ('신규', '증액', '재후원')
)
SELECT parent_campaign AS "상위캠페인명", sub_campaign AS "세부캠페인명", brand AS "브랜드명",
    COUNT(*) AS "총가입자수",
    SUM(CASE WHEN active_months >= 3 THEN 1 ELSE 0 END) AS "3개월유지수",
    ROUND(SUM(CASE WHEN active_months >= 3 THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS "3개월유지율",
    SUM(CASE WHEN active_months >= 6 THEN 1 ELSE 0 END) AS "6개월유지수",
    ROUND(SUM(CASE WHEN active_months >= 6 THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS "6개월유지율",
    SUM(CASE WHEN active_months >= 12 THEN 1 ELSE 0 END) AS "12개월유지수",
    ROUND(SUM(CASE WHEN active_months >= 12 THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 1) AS "12개월유지율",
    AVG("금액") AS "평균개발금액"
FROM member_data
GROUP BY parent_campaign, sub_campaign, brand

## 데이터 재적재 (2026-04-21)
### 대상: 교체_마케팅_26년_디지털대행사.xlsx
- 기존 `추가_마케팅_26년_디지털대행사.xlsx` 로 적재된 26년 데이터 삭제 후 교체 파일로 재적재
- **FACT_DIGITAL_MONTHLY_DEV**: 연도='2026' 삭제 → 재적재
- **FACT_DIGITAL_AD_DETAIL**: 연도='2026' 삭제 → 재적재

In [ ]:
!pip install openpyxl -q

import pandas as pd
import openpyxl
import warnings
warnings.filterwarnings('ignore')
from snowflake.snowpark.context import get_active_session
import tempfile, os, glob

session = get_active_session()
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE DATABASE GN_DW_POC').collect()
session.sql('USE SCHEMA RAW').collect()
session.sql('USE WAREHOUSE COMPUTE_WH').collect()

tmp_dir = tempfile.mkdtemp()
stage_path = '@GN_DW_POC.RAW.STG_POC_DATA'

def dl(filename):
    session.file.get(f"{stage_path}/{filename}", tmp_dir)
    local_path = os.path.join(tmp_dir, filename)
    if not os.path.exists(local_path):
        files = glob.glob(os.path.join(tmp_dir, '*'))
        if files:
            local_path = files[0]
    return local_path

def cs(df, cols):
    for col in cols:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace({'None': None, 'nan': None, 'NaT': None, 'nat': None})
    return df

def clean_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        mask = df[col].isna()
        df[col] = df[col].dt.strftime('%Y-%m-%d %H:%M:%S')
        df.loc[mask, col] = None
    return df

print('Setup complete')

In [ ]:
print('=== 교체_마케팅_26년_디지털대행사.xlsx 재적재 ===')
fp = dl('교체_마케팅_26년_디지털대행사.xlsx')

wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
print(f'Sheets: {wb.sheetnames}')
for sname in wb.sheetnames:
    ws = wb[sname]
    rows = list(ws.iter_rows(max_row=2, values_only=True))
    if rows:
        print(f'  [{sname}] Header: {rows[0]}')
wb.close()

In [ ]:
print('=== Step 1: 기존 26년 디지털 데이터 삭제 ===')

before_monthly = session.sql("SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV").collect()[0]['CNT']
before_detail = session.sql("SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL").collect()[0]['CNT']
print(f'  FACT_DIGITAL_MONTHLY_DEV 삭제 전: {before_monthly:,} rows')
print(f'  FACT_DIGITAL_AD_DETAIL 삭제 전: {before_detail:,} rows')

del_monthly = session.sql("DELETE FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV WHERE \"연도\" = '2026'").collect()
del_detail = session.sql("DELETE FROM GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL WHERE \"연도\" = '2026'").collect()
print(f'  FACT_DIGITAL_MONTHLY_DEV 삭제된 rows: {del_monthly[0][0]}')
print(f'  FACT_DIGITAL_AD_DETAIL 삭제된 rows: {del_detail[0][0]}')

after_monthly = session.sql("SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV").collect()[0]['CNT']
after_detail = session.sql("SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL").collect()[0]['CNT']
print(f'  FACT_DIGITAL_MONTHLY_DEV 삭제 후: {after_monthly:,} rows')
print(f'  FACT_DIGITAL_AD_DETAIL 삭제 후: {after_detail:,} rows')

In [ ]:
print('=== Step 2: 교체 파일에서 디지털 월별 개발 현황 재적재 ===')
df = pd.read_excel(fp, sheet_name='디지털 월별 개발 현황')
kcols = ['연도', '예산절차', '월', '날짜', '월별 개발목표(건)', '월별 개발실적(건)',
         '연목표(건)', '월별 편성예산(원)', '월별 집행예산(원)', '연 광고 예산(원)']
df.columns = kcols
df = cs(df, ['연도', '예산절차', '월', '날짜'])
print(f'  Rows to append: {len(df):,}')

sp_df = session.create_dataframe(df)
sp_df.write.mode('append').save_as_table('GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV')
cnt = session.sql('SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_MONTHLY_DEV').collect()[0]['CNT']
print(f'  FACT_DIGITAL_MONTHLY_DEV 최종: {cnt:,} rows')

In [ ]:
print('=== Step 3: 교체 파일에서 광고운영상세 재적재 ===')
df2 = pd.read_excel(fp, sheet_name='2026년 사단법인 광고운영상세')
kcols2 = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
          '날짜', '주차', '일자', '요일', '캠페인명', '소재', '노출수', '클릭수',
          'GA 광고비', 'GA 전환명수', 'GA 개발건수', '상위캠페인명']
df2.columns = kcols2
df2 = clean_dates(df2, ['날짜'])
for nc in ['노출수', '클릭수', 'GA 광고비', 'GA 전환명수', 'GA 개발건수']:
    df2[nc] = pd.to_numeric(df2[nc], errors='coerce')
str_cols = ['연도', '국내/해외', '사업/사례', '캠페인 유형', '광고유형', '월', '기기', '매체',
            '주차', '일자', '요일', '캠페인명', '소재', '상위캠페인명']
df2 = cs(df2, str_cols)
print(f'  Rows to append: {len(df2):,}')

sp_df2 = session.create_dataframe(df2)
sp_df2.write.mode('append').save_as_table('GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL')
cnt2 = session.sql('SELECT COUNT(*) AS CNT FROM GN_DW_POC.RAW.FACT_DIGITAL_AD_DETAIL').collect()[0]['CNT']
print(f'  FACT_DIGITAL_AD_DETAIL 최종: {cnt2:,} rows')

print('\n=== 교체_마케팅_26년_디지털대행사.xlsx 재적재 완료 ===')